In [ ]:
import os
import glob
import shutil

import numpy as np
import cv2
import matplotlib.pyplot as plt
import yaml
from tqdm.contrib.concurrent import thread_map# or thread_map

import alif_awb
import utils_cam4_revb

%load_ext autoreload
%autoreload 2

In [ ]:
BASE_PTH = "/home/klymy/data/cam4_revB/yolo_dataset"
BASE_DATASET = "batch_H270_16-44-09_margin_10"
PROCESSED_DATASETS = [f"{BASE_DATASET}_wb_alif", f"{BASE_DATASET}_wb_alif_clahe"]

SUBSETS = ["train", "val", "holdout"]

img_paths = glob.glob(os.path.join(BASE_PTH, BASE_DATASET, "images", SUBSETS[2], "*.jpg"))
len(img_paths)

In [ ]:
for i in img_paths[:3]:
    img = cv2.imread(i)

    # Apply white balance and contrast enhancement
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_wb = alif_awb.white_balance_vectorized(img_rgb)
    img_wb_clahe = utils_cam4_revb.clahe(img_wb)
    img_wb_blurred = utils_cam4_revb.blur(img_wb)

    # Display the images
    plt.figure(figsize=(15, 6))
    plt.subplot(1, 4, 1)
    plt.imshow(img_rgb)
    plt.title('Original Image')
    
    plt.subplot(1, 4, 2)
    plt.imshow(img_wb)
    plt.title('White Balance Image')
    
    plt.subplot(1, 4, 3)
    plt.imshow(img_wb_clahe)
    plt.title('White Balance + CLAHE Image')
    
    plt.subplot(1, 4, 4)
    plt.imshow(img_wb_blurred)
    plt.title('White Balance + Blurred Image')
    plt.axis('off')
    plt.tight_layout()
    
    plt.show()

In [ ]:
def process_file(pth):
    img = cv2.imread(pth)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    res = alif_awb.white_balance_vectorized(img_rgb)
    res = cv2.cvtColor(res, cv2.COLOR_RGB2BGR)
    cv2.imwrite(pth, res)

    return 1

def process_file_clahe(pth):
    img = cv2.imread(pth)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    res = alif_awb.white_balance_vectorized(img_rgb)
    res = utils_cam4_revb.clahe(res)
    res = cv2.cvtColor(res, cv2.COLOR_RGB2BGR)
    cv2.imwrite(pth, res)

    return 1

def process_file_blurred(pth):
    img = cv2.imread(pth)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    res = alif_awb.white_balance_vectorized(img_rgb)
    res = utils_cam4_revb.blur(res)
    res = cv2.cvtColor(res, cv2.COLOR_RGB2BGR)
    cv2.imwrite(pth, res)

    return 1

In [ ]:
# for dataset_pth, f in zip(PROCESSED_DATASETS, [process_file, process_file_clahe, process_file_blurred]):
for dataset_pth, f in zip(PROCESSED_DATASETS, [process_file, process_file_clahe]):
    src_pth = os.path.join(BASE_PTH, BASE_DATASET)
    dst_pth = os.path.join(BASE_PTH, dataset_pth)
    if os.path.exists(dst_pth):
        shutil.rmtree(dst_pth)
    shutil.copytree(src_pth, dst_pth)

    cache_files = utils_cam4_revb.get_files_glob(dst_pth, "*.cache")
    for i in cache_files:
        os.remove(i)
    
    # create project file
    utils_cam4_revb.create_project_file(os.path.join(dst_pth, "project.yaml"), dst_pth, {0: "person"})
    utils_cam4_revb.create_project_file(os.path.join(dst_pth, "project_holdout.yaml"), dst_pth, {0: "person"}, "images/holdout")
    
    img_pths = utils_cam4_revb.get_files_glob(os.path.join(dst_pth, "images"), "*.jpg")
    print(len(img_pths))

    r = thread_map(f, img_pths, max_workers=8)
    count = 0
    for i in r:
        count += i

    print(count == len(img_pths))